In [10]:
import numpy as np
import pandas as pd
import random
from chemevo import evolution_V1 as evo

import matplotlib.pyplot as plt
from matplotlib import rcParams
from astropy.table import Table
from astropy.io import fits
import matplotlib.cm as cm
import matplotlib.lines as mlines
import itertools
import random

params = {
   'axes.labelsize': 20,
   'font.size': 15,
   'legend.fontsize': 14,
   'xtick.minor.visible': True,
   'ytick.minor.visible': True,
   'xtick.labelsize': 15,
   'ytick.labelsize': 15,
   'text.usetex': True, #to use TeX in your labels
   'font.family':'serif',
   'axes.titlesize': 20,
   'xtick.direction': 'in',
   'ytick.direction': 'in',
   'xtick.top': True,
   'ytick.right': True,
   'xtick.major.size': 6,
   'xtick.minor.size': 3,
   'ytick.major.size': 6,
   'ytick.minor.size': 3
   }
rcParams.update(params)

import pandas as pd
from astropy.io import fits
from astropy.table import Table

def decode(df):
    """
    Decode DataFrame with byte strings into ordinary strings.
    """
    str_df = df.select_dtypes([object])
    str_df = str_df.stack().str.decode('utf-8').unstack()
    for col in str_df:
        df[col] = str_df[col]
    return df

def fitsrec_to_pandas(fits_rec):
    """
    Convert a FITS_rec object (from astropy.io.fits) to a pandas DataFrame.
    """
    # Convert FITS_rec to astropy Table first
    table = Table(fits_rec)
    
    # Filter out multidimensional columns
    cols = [name for name in table.colnames if len(table[name].shape) <= 1]
    
    # Convert to pandas DataFrame
    df = table[cols].to_pandas()
    
    # Decode byte strings
    df = decode(df)
    
    return df

In [16]:
import pandas as pd
import numpy as np
import glob

# 1. Load the fitted data lines
summary_df = pd.read_csv("data_fitted_line_in_bins_of_mg_h.csv")

# Round the bin centers to avoid floating-point merge errors (e.g., -0.5 vs -0.50000001)
summary_df['mg_h_bin_round'] = summary_df['mg_h_bin_center'].round(3)

# 2. Get all saved model files
model_files = glob.glob("models/*.csv")
chi2_results = []

for file in model_files:
    model_df = pd.read_csv(file)
    
    # Round model bins to match
    model_df['mg_h_bin_round'] = model_df['mg_h_bin'].round(3)
    
    # Merge the model galaxies with the corresponding data line for their bin
    merged_df = pd.merge(
        model_df,
        summary_df,
        on='mg_h_bin_round',
        how='inner'
    )
    
    # Drop rows where the data bin didn't have enough points to fit a line (NaN slopes)
    merged_df = merged_df.dropna(subset=['slope_mn_fe', 'intercept_mn_fe'])
    
    if len(merged_df) == 0:
        continue

    # 3. Calculate Expected Y (Mn/Fe) from the data line
    expected_mn_fe = (merged_df['slope_mn_fe'] * merged_df['fe_mg']) + merged_df['intercept_mn_fe']
    
    # 4. Calculate Chi-Square (Sum of Squared Residuals)
    residuals_sq = (merged_df['mn_fe'] - expected_mn_fe)**2
    chi2_total = residuals_sq.sum()
    n_points = len(residuals_sq)
    
    # Calculate reduced Chi-Square (normalizes by the number of points)
    reduced_chi2 = chi2_total / n_points 

    # Save the results for this global parameter file
    chi2_results.append({
        'file': file,
        'alpha_cc': merged_df['alpha_cc'].iloc[0], # Grab the global parameter
        'alpha_Ia': merged_df['alpha_Ia'].iloc[0], # Grab the global parameter
        'gcc': merged_df['gcc'].iloc[0], # Grab the global parameter
        'gIa/gcc': merged_df['gIa/gcc'].iloc[0], # Grab the global parameter
        'Upsilon': merged_df['Upsilon'].iloc[0], # Grab the global parameter
        'chi2_total': chi2_total,
        'reduced_chi2': reduced_chi2,
        'n_points': n_points
    })

# 5. Convert to DataFrame and find the best fit
results_df = pd.DataFrame(chi2_results)

# Sort by lowest reduced chi2
results_df = results_df.sort_values(by='reduced_chi2').reset_index(drop=True)

print("Top 3 Best Fitting Global Models:")
print(results_df.head(3))

# Isolate the absolute best
best_model = results_df.iloc[0]
print(f"\nThe best fit is {best_model['file']} with alpha_cc = {best_model['alpha_cc']}, alpha_Ia = {best_model['alpha_Ia']}, gcc = {best_model['gcc']}, gIa/gcc = {best_model['gIa/gcc']}, Upsilon = {best_model['Upsilon']}")

Top 3 Best Fitting Global Models:
                    file  alpha_cc  alpha_Ia  gcc  gIa/gcc  Upsilon  \
0  models/alpha_Ia_5.csv       0.6      0.25  0.4        2        1   
1  models/alpha_Ia_6.csv       0.6      0.30  0.4        2        1   
2  models/alpha_Ia_4.csv       0.6      0.20  0.4        2        1   

   chi2_total  reduced_chi2  n_points  
0    4.312876      0.002883      1496  
1    4.410451      0.002948      1496  
2    4.700980      0.003142      1496  

The best fit is models/alpha_Ia_5.csv with alpha_cc = 0.6, alpha_Ia = 0.25, gcc = 0.4, gIa/gcc = 2, Upsilon = 1


# MCMC model

In [19]:
# Star formation histories
t_array = np.linspace(0.0001,14,int(1e3))
dt = t_array[1] - t_array[0]

def rise_fall(t, tau_1, tau_2):
    return 1 * (1 - np.exp(-t/tau_1)) * np.exp(-t/tau_2)

m_g_array_const = np.ones(len(t_array))
tau_sfh = 6 #Gyrs
m_g_array_exp = 1 * np.exp(-t_array/tau_sfh)
m_g_array_exp /= np.trapezoid(m_g_array_exp, t_array)

m_g_array_rise_fall_1 = rise_fall(t_array, 2, 8)
m_g_array_rise_fall_1 /= np.trapezoid(m_g_array_rise_fall_1, t_array)

m_g_array_rise_fall_2 = rise_fall(t_array, 4, 30)
m_g_array_rise_fall_2 /= np.trapezoid(m_g_array_rise_fall_2, t_array)

m_g_array_rise_fall_3 = rise_fall(t_array, 6, 12)
m_g_array_rise_fall_3 /= np.trapezoid(m_g_array_rise_fall_3, t_array)

#sfrs = [m_g_array_const, m_g_array_exp, m_g_array_rise_fall_1, m_g_array_rise_fall_2, m_g_array_rise_fall_3]


# Default met dep params
gcc_Mn_def = 0.40
gcc_ratio_def = 2
alpha_cc_def = 0.60
alpha_ia_def = 0.30


# Etas and Tau*s 
#etas = np.logspace(-2, 1, 10)
#tau_stars = np.linspace(0.5, 6, 10)


etas = np.logspace(-2, 1, 2)
tau_stars = np.linspace(0.5, 6, 2)
sfrs = [m_g_array_const, m_g_array_rise_fall_1]


all_params = list(itertools.product(etas, tau_stars, sfrs))

print("n samples in each met dep param choice = ", len(all_params))

# Get endpoint indexes
def find_endpoints(galaxies, bin_centers):
    endpoints_gals = []
    for bin_cent in bin_centers:
        endpoints = []
        for gal in galaxies:
            mg_h = gal.Mg_H
            crossed = False
            for indx in range(len(mg_h)):
                mg_h_at_time = mg_h[indx]
                if mg_h_at_time > bin_cent:
                    endpoints.append(indx)
                    crossed = True
                    break
            if not crossed:
                endpoints.append(-1) # Critical fix to maintain 1:1 mapping with galaxies
        endpoints_gals.append(np.array(endpoints))
    return endpoints_gals

mg_h_big_bin_edges = np.arange(-0.7, 0.50, 0.20) 
mg_h_big_bin_centers = 0.5 * (mg_h_big_bin_edges[:-1] + mg_h_big_bin_edges[1:])

n samples in each met dep param choice =  8


In [ ]:
import numpy as np
import pandas as pd
import emcee
from chemevo import evolution_V1 as evo

# --- 1. Load your observational data ---
summary_df = pd.read_csv("data_fitted_line_in_bins_of_mg_h.csv")

# --- 2. Update your galaxy generator to accept Upsilon ---
def get_sampled_gals(params, alpha_cc, alpha_Ia, g_ratio, gcc, Upsilon):
    return [
        evo.Galaxy(
            t_array=t_array, m_g_array=sfr, tau_star=tau_star,
            eta=eta, Upsilon=Upsilon, yields_ref="W2024,moreFe",
            g_cc_Mn=gcc, g_ratio_Mn=g_ratio,
            alpha_cc_Mn=alpha_cc, alpha_Ia_Mn=alpha_Ia
        ) for eta, tau_star, sfr in params
    ]

# --- 3. Define the MCMC Functions ---

def log_prior(theta):
    """Sets the boundaries for your 5 parameters."""
    g_ratio, g_cc, alpha_Ia, alpha_cc, Upsilon = theta
    
    # Check if all parameters are within your desired ranges
    if (0.1 < g_ratio < 3.0) and (0.1 < g_cc < 0.6) and \
       (0.0 <= alpha_Ia <= 1.0) and (0.0 <= alpha_cc <= 1.0) and \
       (0.5 < Upsilon < 3.0):
        return 0.0 # Allowed
    return -np.inf   # Disallowed (Zero probability)

def log_likelihood(theta, data_summary):
    """Generates the model and calculates Chi^2."""
    g_ratio, g_cc, alpha_Ia, alpha_cc, Upsilon = theta
    
    # 1. Generate the models for this specific step
    gals_list = get_sampled_gals(all_params, alpha_cc, alpha_Ia, g_ratio, g_cc, Upsilon)
    gals_endpoints = find_endpoints(gals_list, mg_h_big_bin_centers)
    
    # 2. Extract the model data (Simplified extraction)
    model_data = []
    for i in range(len(mg_h_big_bin_centers)):
        current_mg_h = mg_h_big_bin_centers[i]
        for gal, idx in zip(gals_list, gals_endpoints[i]):
            if idx != -1:
                model_data.append({'mn_fe': gal.Mn_Fe[idx], 'fe_mg': gal.Fe_Mg[idx], 'mg_h_bin': current_mg_h})
    
    model_df = pd.DataFrame(model_data)
    if model_df.empty:
        return -np.inf
    
    # 3. Calculate Chi-Square against the data line
    merged_df = pd.merge(model_df.round(3), data_summary.round(3), left_on='mg_h_bin', right_on='mg_h_bin_center')
    merged_df = merged_df.dropna(subset=['slope_mn_fe', 'intercept_mn_fe'])
    
    expected_mn_fe = (merged_df['slope_mn_fe'] * merged_df['fe_mg']) + merged_df['intercept_mn_fe']
    chi2 = np.sum((merged_df['mn_fe'] - expected_mn_fe)**2)
    
    # emcee maximizes the log-likelihood, so we return -0.5 * chi^2
    return -0.5 * chi2

def log_probability(theta, data_summary):
    """Combines prior bounds and likelihood."""
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data_summary)

# --- 4. Run the MCMC ---

ndim = 5 # Number of parameters
nwalkers = 32 # Number of MCMC chains to run simultaneously

# Start walkers near the middle of your parameter bounds
initial_guess = [1.5, 0.35, 0.5, 0.5, 1.75]
# Add a tiny bit of random noise so walkers don't start in the exact same spot
pos = initial_guess + 1e-4 * np.random.randn(nwalkers, ndim)

print("Starting MCMC...")
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(summary_df,))

# Run for 1000 steps (Total models run = nwalkers * steps = 32,000)
sampler.run_mcmc(pos, 1000, progress=True)

# Save the results
flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)
np.save("mcmc_chain.npy", flat_samples)

Starting MCMC...


  7%|▋         | 68/1000 [1:59:56<50:05:29, 193.49s/it] 